# 25. Kontras tambahan ablasi dan kepekaan mask kosong
**Penelitian:** Deteksi anemia dari citra pucat (telapak tangan, kuku, konjungtiva) dengan data pooling multi-situs.

**Tujuan:** menghitung kontras berpasangan tambahan pada ablasi (efek filter intensitas dengan dan tanpa segmentasi, Otsu vs K-Means, segmentasi tanpa filter) dan analisis kepekaan terhadap 38 citra bermask kosong (fitur nol diganti fitur tanpa filter), memakai fitur dan lipatan validasi silang 5x5 yang sama dengan Notebook 23. Membaca `fitur_ablasi.csv` dari folder hasil_revisi (keluaran Notebook 23); tidak membaca citra sehingga cepat.

**Masukan:** Dataset (Mendeley Data): Palm doi:10.17632/ccr8cm22vz.1, Nail doi:10.17632/2xx4j3kjg2.1, Conjunctiva doi:10.17632/nt7r8hv2pz.1. Letakkan dalam folder `Dataset_Multisite/{Palm,Nail,Conjunctiva}`.

**Keluaran (folder `hasil_revisi`):** `ablasi_kontras_tambahan.csv`, `ablasi_mask_kosong.csv`.

**Kaitan dengan naskah:** Bagian 3.4 (efek filter dan segmentasi) dan analisis kepekaan di Bagian 2.2.

**Cara menjalankan:** Colab, Runtime > Run all. CPU sudah cukup; GPU tidak diperlukan. Seed acak = 42 dan urutan file diurutkan (`sorted`) sehingga hasil dapat direproduksi pada versi pustaka yang dicetak di sel kedua.

In [1]:
!pip install lightgbm -q

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

Mounted at /content/drive


In [3]:
# ===== FUNGSI BERSAMA (identik di keempat notebook) =====
import os, re, hashlib, warnings
import numpy as np, pandas as pd, cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

# ---------- KONFIGURASI (ubah hanya bagian ini) ----------
SEED = 42
BASE = '/content/drive/MyDrive/Dataset_Multisite'   # folder berisi subfolder Palm/, Nail/, Conjunctiva/
OUT  = '/content/drive/MyDrive/hasil_revisi'          # folder keluaran (CSV)
SITES = {'Palm': 0, 'Nail': 1, 'Conjunctiva': 2}
# Filter piksel memeriksa kanal L, R, dan G. Kanal B ikut difilter jika True
# (Appiahene et al. 2023 menuliskan kondisi min < R=G=B < max).
FILTER_INCLUDE_B = False
# True: buang juga citra bernama varian dari citra lain (akhiran FV/RA atau spasi+angka) yang citra dasarnya ada.
BUANG_VARIAN = True
os.makedirs(OUT, exist_ok=True)

# ---------- DAFTAR FILE ----------
def parse_label(name):
    """Label dari awalan nama file (Anemic / Non-Anemic); nama yang mengandung tanda kurung (salinan bernomor) dan awalan tidak dikenali dilewati."""
    if '(' in name: return None
    c = name.lower().replace(' ', '')
    if c.startswith('non-anemic'): return 0
    if c.startswith('anemic'): return 1
    return None

def list_used_files():
    rows = []
    for site, sid in SITES.items():
        d = os.path.join(BASE, site)
        for n in sorted(os.listdir(d)):          # sorted -> urutan deterministik
            lab = parse_label(n)
            if lab is not None:
                rows.append(dict(site=site, site_id=sid, file=n, path=os.path.join(d, n), label=lab))
    return pd.DataFrame(rows)

def md5(path):
    with open(path, 'rb') as f: return hashlib.md5(f.read()).hexdigest()

# ---------- EKSTRAKSI FITUR ----------
def feat_original(img, lo=50, hi=200):
    """Fitur [mean a*, mean b*, mean G] dengan filter piksel. Return (fitur, mask_kosong)."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab); R, G, B = cv2.split(rgb)
    m = (L > lo) & (L < hi) & (R > lo) & (R < hi) & (G > lo) & (G < hi)
    if FILTER_INCLUDE_B: m &= (B > lo) & (B < hi)
    if m.any(): return [a[m].mean(), b[m].mean(), G[m].mean()], False
    return [0.0, 0.0, 0.0], True

def feat_otsu(img):
    """Fitur dari piksel foreground Otsu (grayscale). Tanpa filter L/R/G. Return (fitur, mask_kosong)."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mk = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = mk > 0
    if not m.any(): return [0.0, 0.0, 0.0], True
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    _, a, b = cv2.split(lab); G = rgb[:, :, 1]
    return [a[m].mean(), b[m].mean(), G[m].mean()], False

# ---------- PEMBAGIAN DATA & EVALUASI ----------
def split_idx(y, scheme, seed=SEED):
    """Return (train, val, test) berupa indeks. Skema: '80:20', '80:10:10', '70:10:20'."""
    idx = np.arange(len(y))
    if scheme == '80:20':
        tr, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        return tr, None, te
    if scheme == '80:10:10':
        tr, tmp = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
        return tr, va, te
    if scheme == '70:10:20':
        trva, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        tr, va = train_test_split(trva, test_size=0.125, stratify=y[trva], random_state=seed)
        return tr, va, te
    raise ValueError(scheme)

def metrics_from(yte, pred, proba):
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(yte), 'TP+TN+FP+FN harus sama dengan jumlah data uji'
    two = len(set(yte)) == 2
    return dict(n_test=len(yte), n_test_anemic=int((yte == 1).sum()), TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                acc=100*(tp+tn)/len(yte),
                sens=100*tp/(tp+fn) if tp+fn else np.nan,
                spec=100*tn/(tn+fp) if tn+fp else np.nan,
                f1=100*f1_score(yte, pred, zero_division=0),
                auc=100*roc_auc_score(yte, proba) if two else np.nan,
                baseline_mayoritas=100*max((yte == 1).mean(), (yte == 0).mean()))

def fit_lgbm(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    m = LGBMClassifier(n_estimators=100, random_state=SEED, verbose=-1).fit(sc.transform(Xtr), ytr)
    return sc, m

def predict_lgbm(sc, m, X):
    proba = m.predict_proba(sc.transform(X))[:, 1]
    return (proba >= 0.5).astype(int), proba

def tandai_dibuang(df):
    """Array boolean (urut baris): True bila baris dibuang pada pembersihan = salinan identik MD5 (satu dipertahankan),
    kedua file pada pasangan identik berlabel berbeda, dan (bila BUANG_VARIAN) citra varian FV/RA. df: indeks 0..n-1 dan kolom md5."""
    konflik = df.groupby('md5').label.transform('nunique') > 1
    dibuang = np.array(df.duplicated('md5', keep='first') | konflik, dtype=bool)   # salinan yang dapat diubah
    if BUANG_VARIAN:
        posisi = np.flatnonzero(~dibuang)
        for a, b in pasangan_varian(df.iloc[posisi].reset_index(drop=True)): dibuang[posisi[a]] = True
    return dibuang

def bersihkan(df):
    """Data bersih untuk semua eksperimen (satu definisi tunggal)."""
    df = df.reset_index(drop=True).copy(); df['md5'] = df['path'].map(md5)
    return df[~tandai_dibuang(df)].reset_index(drop=True)

def cek_kebocoran(md5s, tr, va, te, wajib=True):
    """Pemeriksaan kebocoran: indeks latih/validasi/uji harus saling lepas dan tidak ada citra identik (MD5) lintas partisi.
    Return jumlah citra identik lintas partisi, mis. {'latih-uji': 0}. wajib=True -> hentikan program bila ada pelanggaran."""
    parts = {'latih': set(tr), 'uji': set(te)}
    if va is not None: parts['val'] = set(va)
    nm = sorted(parts)
    for i, a in enumerate(nm):
        for b in nm[i + 1:]:
            assert not (parts[a] & parts[b]), f'Indeks tumpang tindih: {a}-{b}'
    m = {k: set(np.asarray(md5s)[sorted(v)]) for k, v in parts.items()}
    bocor = {f'{a}-{b}': len(m[a] & m[b]) for i, a in enumerate(nm) for b in nm[i + 1:]}
    if wajib: assert all(v == 0 for v in bocor.values()), f'Citra identik lintas partisi: {bocor}'
    return bocor

def pasangan_varian(df):
    """Pasangan (indeks varian, indeks citra dasar) pada df (indeks posisi 0..n-1): nama berakhiran FV/RA atau spasi+angka
    yang nama dasarnya ada pada situs yang sama (mis. Anemic-001FV dengan Anemic-001)."""
    stem = df.file.str.replace(r'\.[A-Za-z]+$', '', regex=True)
    base = stem.str.replace(r'\s\d+$', '', regex=True).str.replace(r'(FV|RA)$', '', regex=True)
    key = {(s, t): i for i, (s, t) in enumerate(zip(df.site, stem))}
    return [(i, key[(s, b)]) for i, (s, t, b) in enumerate(zip(df.site, stem, base)) if b != t and (s, b) in key]

def cek_varian(pasangan, tr, va, te):
    """Jumlah pasangan varian-dasar yang terpisah di partisi berbeda (informasi; idealnya 0)."""
    part = {}
    for ids, nm in [(tr, 'latih'), (va, 'val'), (te, 'uji')]:
        if ids is not None:
            for i in ids: part[i] = nm
    return sum(1 for a, b in pasangan if a in part and b in part and part[a] != part[b])

In [4]:
import sys, numpy, pandas, cv2, sklearn, lightgbm
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| OpenCV', cv2.__version__, '| scikit-learn', sklearn.__version__, '| LightGBM', lightgbm.__version__, '| SEED', SEED)

Python 3.13.16 | numpy 2.4.6 | pandas 2.2.3 | OpenCV 4.14.0 | scikit-learn 1.6.1 | LightGBM 4.6.0 | SEED 42


In [5]:
# ===== KONTRAS TAMBAHAN ABLASI DAN KEPEKAAN MASK KOSONG (membaca fitur_ablasi.csv dari Notebook 23; tanpa membaca citra) =====
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold
N_SPLITS, N_REPEATS = 5, 5
f = pd.read_csv(os.path.join(OUT, 'fitur_ablasi.csv'))
y = f.label.to_numpy(); site = f.site.map(SITES).to_numpy(float)
def kol(a): return f[[f'{a}|a', f'{a}|b', f'{a}|G']].to_numpy(float)
def kosong(a): return (kol(a) == 0).all(axis=1)
REF, NOFIL, OTSU, OTSUF, KM, KMF = 'Tanpa segmentasi + filter (Original)', 'Tanpa segmentasi, tanpa filter', 'Otsu', 'Otsu + filter', 'K-Means', 'K-Means + filter'

# jumlah citra bermask kosong (fitur nol) per perlakuan dan situs
mk = []
for a in sorted({c.split('|')[0] for c in f.columns if '|' in c}):
    k = kosong(a)
    if k.any(): mk.append(dict(perlakuan=a, total=int(k.sum()), anemic=int(y[k].sum()), **{s: int((f.site[k] == s).sum()) for s in SITES}))
mk = pd.DataFrame(mk); mk.to_csv(os.path.join(OUT, 'ablasi_mask_kosong.csv'), index=False); print(mk.to_string(index=False))

def ganti(a, fb):                     # fitur dengan mask kosong digantikan fitur tanpa filter
    X = kol(a).copy(); m = kosong(a); X[m] = kol(fb)[m]; return X
FIT = {REF: kol(REF), NOFIL: kol(NOFIL), OTSU: kol(OTSU), OTSUF: kol(OTSUF), KM: kol(KM), KMF: kol(KMF),
       REF + ' [kosong diganti fitur tanpa filter]': ganti(REF, NOFIL), OTSUF + ' [kosong diganti Otsu]': ganti(OTSUF, OTSU), KMF + ' [kosong diganti K-Means]': ganti(KMF, KM)}
strat = (site * 2 + y).astype(int)
folds = list(RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED).split(np.zeros(len(y)), strat))
rasio = float(np.mean([len(te) / len(tr) for tr, te in folds]))
def cv(X):
    acc, auc = [], []
    for tr, te in folds:
        sc, m = fit_lgbm(X[tr], y[tr]); pr, pb = predict_lgbm(sc, m, X[te]); acc.append(100 * (pr == y[te]).mean()); auc.append(100 * roc_auc_score(y[te], pb))
    return {'acc': np.array(acc), 'auc': np.array(auc)}
R = {(ws, k): cv(np.column_stack([X, site]) if ws else X) for ws in [True, False] for k, X in FIT.items()}
def banding(d):                       # uji t terkoreksi Nadeau-Bengio untuk selisih berpasangan antar-lipatan
    k = len(d); m = d.mean(); v = d.var(ddof=1)
    if v < 1e-12: return m, np.nan, np.nan, np.nan
    se = np.sqrt((1 / k + rasio) * v); h = stats.t.ppf(0.975, k - 1) * se
    return m, m - h, m + h, 2 * stats.t.sf(abs(m / se), k - 1)
K = [('Segmentasi tanpa filter: Otsu - tanpa segmentasi (tanpa filter)', OTSU, NOFIL), ('Segmentasi tanpa filter: K-Means - tanpa segmentasi (tanpa filter)', KM, NOFIL),
     ('Efek filter pada Otsu: Otsu+filter - Otsu', OTSUF, OTSU), ('Efek filter pada K-Means: K-Means+filter - K-Means', KMF, KM),
     ('Efek filter tanpa segmentasi: filter - tanpa filter', REF, NOFIL), ('Otsu - K-Means (tanpa filter)', OTSU, KM),
     ('Kepekaan: Original (kosong diganti) - Original (kosong nol)', REF + ' [kosong diganti fitur tanpa filter]', REF),
     ('Kepekaan: Otsu - Original (kosong diganti)', OTSU, REF + ' [kosong diganti fitur tanpa filter]'),
     ('Kepekaan: K-Means - Original (kosong diganti)', KM, REF + ' [kosong diganti fitur tanpa filter]'),
     ('Kepekaan: Otsu+filter (kosong diganti) - Otsu+filter (kosong nol)', OTSUF + ' [kosong diganti Otsu]', OTSUF)]
rows = []
for ws in [True, False]:
    for nm, a, b in K:
        for m in ['auc', 'acc']:
            mu, lo, hi, p = banding(R[(ws, a)][m] - R[(ws, b)][m])
            rows.append(dict(site_id='dengan' if ws else 'tanpa', kontras=nm, metrik=m, selisih=mu, ci_bawah=lo, ci_atas=hi, p_terkoreksi=p))
out = pd.DataFrame(rows).round(4); out.to_csv(os.path.join(OUT, 'ablasi_kontras_tambahan.csv'), index=False)
print(out[out.metrik == 'auc'].to_string(index=False))

                           perlakuan  total  anemic  Palm  Nail  Conjunctiva
             CLAHE (mask citra asli)     38      26     2    23           13
             CLAHE (mask dari hasil)     62      41     5    23           34
                    K-Means + filter    122      74    86    23           13
                       Otsu + filter    130      77    94    23           13
Tanpa segmentasi + filter (Original)     38      26     2    23           13
          WB+CLAHE (mask citra asli)     38      26     2    23           13
          WB+CLAHE (mask dari hasil)     28      18     1    16           11
     White Balance (mask citra asli)     38      26     2    23           13
     White Balance (mask dari hasil)      4       3     0     3            1
site_id                                                            kontras metrik  selisih  ci_bawah  ci_atas  p_terkoreksi
 dengan    Segmentasi tanpa filter: Otsu - tanpa segmentasi (tanpa filter)    auc   4.3744    1.3549   7.3